In [2]:
import sys
print(sys.executable)

c:\Users\NIHAL\Desktop\classification_project_items\.venv\Scripts\python.exe


# Diabetes Classification and Clustering

 1. Objective
Explore the diabetes dataset, build and evaluate classification models,
and compare the known classes with groups discovered through clustering.

 2. Environment verification
Confirm that this notebook uses the project's Python virtual environment.

In [4]:
from pathlib import Path 
print(Path.cwd())

c:\Users\NIHAL\Desktop\classification_project_items\notebooks


In [5]:
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "dataset.xlsx"

print(data_path)
print("File exists:", data_path.is_file())

c:\Users\NIHAL\Desktop\classification_project_items\data\raw\dataset.xlsx
File exists: True


importing data and its library 

In [6]:
import pandas as pd 

df = pd.read_excel(data_path)
df.head() 

,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [7]:
print("Rows and columns:", df.shape)
df.info()

Rows and columns: (1000, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   No_Pation  1000 non-null   int64  
 2   Gender     1000 non-null   object 
 3   AGE        1000 non-null   int64  
 4   Urea       1000 non-null   float64
 5   Cr         1000 non-null   int64  
 6   HbA1c      1000 non-null   float64
 7   Chol       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASS      1000 non-null   object 
dtypes: float64(8), int64(4), object(2)
memory usage: 109.5+ KB


In [8]:
for column in ["Gender", "CLASS"]:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))
    print("Exact values:", [repr(value) for value in df[column].unique()])


Gender:
Gender
M    565
F    434
f      1
Name: count, dtype: int64
Exact values: ["'F'", "'M'", "'f'"]

CLASS:
CLASS
Y     840
N     102
P      53
Y       4
N       1
Name: count, dtype: int64
Exact values: ["'N'", "'N '", "'P'", "'Y'", "'Y '"]


In [9]:
df_clean = df.copy()

for column in ["Gender", "CLASS"]:
    df_clean[column] = df_clean[column].str.strip().str.upper()

In [11]:
print("Shape unchanged:", df_clean.shape == df.shape)

for column in ["Gender", "CLASS"]:
    print(f"\n{column}:")
    print(df_clean[column].value_counts(dropna=False))
    print("Exact values:", [repr(value) for value in df_clean[column].unique()])

Shape unchanged: True

Gender:
Gender
M    565
F    435
Name: count, dtype: int64
Exact values: ["'F'", "'M'"]

CLASS:
CLASS
Y    844
N    103
P     53
Name: count, dtype: int64
Exact values: ["'N'", "'P'", "'Y'"]


Standardized capitalization and removed surrounding whitespace in Gender and CLASS. All 1,000 rows were preserved. Class counts are Y: 844, N: 103, and P: 53, indicating substantial class imbalance.

In [12]:
record_columns = df_clean.columns.drop(["ID", "No_Pation"])

print("Exact duplicate rows:", df_clean.duplicated().sum())
print(
    "Repeated records excluding identifiers:",
    df_clean.duplicated(subset=record_columns).sum()
)

Exact duplicate rows: 0
Repeated records excluding identifiers: 174


In [13]:
repeated_mask = df_clean.duplicated(
    subset=record_columns,
    keep=False
)

df_clean.loc[repeated_mask].sort_values(
    by=list(record_columns)
).head(10)

,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
12,82,46815,F,30,3.0,42,4.1,4.9,1.3,1.2,3.2,0.5,22.0,N
17,676,87654,F,30,3.0,42,4.1,4.9,1.3,1.2,3.2,0.5,22.0,N
129,466,26665,F,30,5.7,53,6.0,5.4,1.7,1.4,3.3,0.7,22.0,P
131,688,87660,F,30,5.7,53,6.0,5.4,1.7,1.4,3.3,0.7,22.0,P
205,140,24018,F,33,1.8,31,7.4,3.7,1.2,1.6,1.5,0.5,31.5,Y
211,453,48112,F,33,1.8,31,7.4,3.7,1.2,1.6,1.5,0.5,31.5,Y
121,399,47513,F,39,3.0,38,6.4,4.7,1.3,1.1,3.1,0.6,22.0,P
125,690,87661,F,39,3.0,38,6.4,4.7,1.3,1.1,3.1,0.6,22.0,P
21,97,12744,F,42,5.0,73,4.5,6.2,1.0,1.1,4.6,0.4,24.0,N
33,678,87655,F,42,5.0,73,4.5,6.2,1.0,1.1,4.6,0.4,24.0,N


In [14]:
for column in ["ID", "No_Pation"]:
    print(f"\n{column}:")
    print("Unique values:", df_clean[column].nunique())
    print("Repeated entries after first:", df_clean[column].duplicated().sum())


ID:
Unique values: 800
Repeated entries after first: 200

No_Pation:
Unique values: 961
Repeated entries after first: 39


In [15]:
patient_repeat_mask = df_clean["No_Pation"].duplicated(keep=False)

df_clean.loc[patient_repeat_mask].sort_values("No_Pation").head(10)

,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
697,576,234,M,66,4.10,63,10.1,3.6,5.1,0.9,2.5,0.9,30.0,Y
599,584,234,M,59,4.20,56,10.5,4.9,2.1,1.1,2.5,0.9,28.0,Y
664,554,345,M,55,4.40,56,10.0,4.9,2.1,1.1,2.5,0.9,28.0,Y
947,122,345,M,60,3.30,59,7.6,3.5,1.0,1.3,1.5,0.7,30.0,Y
868,28,856,F,55,8.50,59,7.6,3.3,1.7,0.9,1.7,0.8,32.0,Y
672,586,856,F,61,4.30,56,12.1,4.4,2.0,1.0,2.5,0.9,29.0,Y
425,601,2345,M,55,4.16,60,7.6,6.5,2.1,1.3,4.3,14.5,34.5,Y
892,55,2345,M,55,26.40,228,7.1,4.1,1.9,1.3,2.0,0.9,27.0,Y
781,548,14389,F,60,4.10,56,13.1,4.4,2.0,1.0,2.5,0.9,29.0,Y
171,231,14389,F,50,4.00,56,13.7,4.4,2.0,1.0,2.5,0.9,29.0,Y


Neither identifier is unique. Repeated No_Pation values can have different measurements. Patient identity is uncertain, so identifier-based deletion is not justified. Identifiers will be excluded from model inputs, and possible patient overlap will be considered when designing evaluation.

In [16]:
numeric_columns = df_clean.select_dtypes(include="number").columns
measurement_columns = numeric_columns.drop(["ID", "No_Pation"])

df_clean[measurement_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE,1000.0,53.528000,8.799241,20.0,51.0,55.0,59.0,79.00
Urea,1000.0,5.124743,2.935165,0.5,3.7,4.6,5.7,38.90
Cr,1000.0,68.943000,59.984747,6.0,48.0,60.0,73.0,800.00
HbA1c,1000.0,8.281160,2.534003,0.9,6.5,8.0,10.2,16.00
Chol,1000.0,4.862820,1.301738,0.0,4.0,4.8,5.6,10.30
TG,1000.0,2.349610,1.401176,0.3,1.5,2.0,2.9,13.80
HDL,1000.0,1.204750,0.660414,0.2,0.9,1.1,1.3,9.90
LDL,1000.0,2.609790,1.115102,0.3,1.8,2.5,3.3,9.90
VLDL,1000.0,1.854700,3.663599,0.1,0.7,0.9,1.5,35.00
BMI,1000.0,29.578020,4.962388,19.0,26.0,30.0,33.0,47.75


In [17]:
zero_chol_mask = df_clean["Chol"] == 0

print("Rows with zero cholesterol:", zero_chol_mask.sum())
df_clean.loc[zero_chol_mask]

Rows with zero cholesterol: 1


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
99,186,454316,M,77,5.0,106,5.4,0.0,2.8,0.8,1.8,0.7,19.0,N


In [18]:
for column in ["Cr", "HDL", "VLDL"]:
    print(f"\nHighest values for {column}:")
    display(df_clean.nlargest(3, column))

print("\nLowest HbA1c values:")
display(df_clean.nsmallest(3, "HbA1c"))


Highest values for Cr:


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
273,1,34325,M,58,20.8,800,9.1,6.6,2.9,1.1,4.3,1.3,33.0,Y
283,266,24060,M,58,20.8,800,9.1,6.6,2.9,1.1,4.3,1.3,33.0,Y
846,1,34325,M,56,20.8,800,9.0,4.6,2.0,1.2,2.5,0.9,35.0,Y



Highest values for HDL:


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
752,154,34522,F,56,4.5,57,11.8,3.9,3.1,9.9,1.8,1.4,33.0,Y
816,181,45511,F,55,5.5,50,12.2,6.3,2.3,9.0,4.4,1.0,31.0,Y
412,443,24119,F,52,6.4,88,8.5,8.5,0.8,6.6,1.3,1.3,33.0,Y



Highest values for VLDL:


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
526,604,876543,M,66,5.50,73,10.9,9.3,5.1,0.9,6.4,35.0,27.75,Y
565,624,356355,M,55,6.66,61,6.7,2.5,4.9,0.8,0.5,33.6,29.00,Y
242,203,24039,M,55,5.00,76,10.2,5.6,4.6,0.8,2.9,31.8,33.90,Y



Lowest HbA1c values:


,ID,No_Pation,Gender,AGE,Urea,Cr,HbA1c,Chol,TG,HDL,LDL,VLDL,BMI,CLASS
80,138,45399,F,57,4.6,97,0.9,3.2,1.3,0.9,3.0,1.1,22.0,N
92,161,45406,F,60,7.5,70,0.9,3.4,5.3,1.1,3.6,0.8,24.0,N
98,182,454316,M,60,7.5,70,0.9,3.4,5.3,1.1,3.6,0.8,24.0,N


Numeric inspection identified extreme values in Cr, HDL, and VLDL, low HbA1c values, and one zero cholesterol value. Their validity and measurement units require verification. Values are retained pending a documented handling decision. Repeated identifiers do not consistently correspond to identical records.

## Data audit findings

- Dataset: 1,000 rows and 14 columns.
- No missing values were detected by pandas.
- Standardized capitalization and surrounding whitespace in Gender and CLASS.
- Cleaned class counts: Y = 844, N = 103, P = 53.
- Found 0 exact duplicate rows.
- Found 174 repeated rows after the first occurrence when excluding identifiers.
- Neither ID nor No_Pation is unique; patient identity remains uncertain.
- Flagged zero cholesterol and unusual numeric extremes for investigation.
- Retained all rows and numeric values during this initial audit.
- Identifiers will be excluded from model inputs.
- Identical feature records must remain together during evaluation splitting.

In [19]:
feature_columns = df_clean.columns.drop(["ID", "No_Pation", "CLASS"])

print("Input columns:", feature_columns.tolist())
print(
    "Repeated feature rows:",
    df_clean.duplicated(subset=feature_columns).sum()
)

Input columns: ['Gender', 'AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL', 'LDL', 'VLDL', 'BMI']
Repeated feature rows: 174


In [20]:
labels_per_group = df_clean.groupby(
    feature_columns.tolist(),
    dropna=False
)["CLASS"].nunique()

print("Feature groups with conflicting labels:", (labels_per_group > 1).sum())

Feature groups with conflicting labels: 0
